# DS 208 · Week 12 — Integrative project
## Functions, tests and a pipeline that runs end to end

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 11 — The project’s functions and tests run here


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 13 — Early failure is preferable to a wrong result


Expected output:

```
missing data/flood_raw.csv: see README
```


This cell **fails on purpose**. It is wrapped in `try`/`except` so the notebook keeps running; the error is printed instead.


In [ ]:
try:
    # fails on purpose: read the last line
    path = "data/flood_raw.csv"          # not there yet on a fresh clone

    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        raise SystemExit(f"missing {path}: see README")

    assert len(df) > 0, "loaded an empty file"
    assert "budget" in df.columns, "schema changed"
except SystemExit as e:
    print(e)
except Exception as e:
    print(type(e).__name__ + ":", e)

### Slide 18 — Extracting repeated cells into functions


Expected output:

```
budget                      int64
start_date         datetime64[us]
completion_date    datetime64[us]
duration                    int64
dtype: object
```


In [ ]:
# src/clean.py
def coerce_types(df):
    out = df.copy()                     # never modify the caller's frame
    out["budget"] = pd.to_numeric(out.budget, errors="coerce")
    for col in ("start_date", "completion_date"):
        out[col] = pd.to_datetime(out[col], errors="coerce")
    return out

def add_duration(df):
    out = df.copy()
    out["duration"] = (out.completion_date - out.start_date).dt.days
    return out

demo = pd.DataFrame({"budget": ["5000000"], "start_date": ["2024-01-10"],
                     "completion_date": ["2024-06-30"]})
print(add_duration(coerce_types(demo)).dtypes)

### Slide 19 — Docstrings explain why; code shows what


Expected output:

```
2 kept, 1 dropped
```


In [ ]:
def drop_unusable(df, subset):
    """Drop rows missing any of `subset`, and report how many went.

    Deliberately NOT a bare df.dropna(): that removes rows for reasons
    unrelated to the question and silently changes which population
    the analysis is about.
    """
    before = len(df)
    out = df.dropna(subset=subset)
    return out, before - len(out)

demo = pd.DataFrame({"budget": [1.0, None, 3.0], "notes": [None, "x", None]})
kept, dropped = drop_unusable(demo, ["budget"])
print(len(kept), "kept,", dropped, "dropped")

### Slide 22 — Each row exists to exercise one case


Expected output:

```
id   budget  start_date completion_date
0  A1  1000000  2024-01-10      2024-06-30
1  A2  2500000  2024-02-01             NaN
2  A3      NaN  2024-03-05      2024-09-01
3  A4   800000  2016-05-23      2016-05-06
4  A5      n/a  2024-04-01      2024-08-15
```


In [ ]:
RAW = pd.DataFrame([
    {"id": "A1", "budget": "1000000", "start_date": "2024-01-10", "completion_date": "2024-06-30"},  # normal
    {"id": "A2", "budget": "2500000", "start_date": "2024-02-01", "completion_date": ""},            # unfinished
    {"id": "A3", "budget": "",        "start_date": "2024-03-05", "completion_date": "2024-09-01"},  # no budget
    {"id": "A4", "budget": "800000",  "start_date": "2016-05-23", "completion_date": "2016-05-06"},  # impossible
    {"id": "A5", "budget": "n/a",     "start_date": "2024-04-01", "completion_date": "2024-08-15"},  # unparseable
]).replace("", pd.NA)
print(RAW)

### Slide 23 — Testing invariants rather than implementation


**Exercise.**

1. Predict whether both tests pass.
2. Run the cell.
3. In coerce_types (on the Run / Edit copy), delete errors="coerce" and run again. What happens?


In [ ]:
# coerce_types, add_duration, RAW: from the previous slides
def test_coerce_counts_failures():
    out = coerce_types(RAW)
    assert out.budget.isna().sum() == 2      # A3 empty + A5 "n/a"

def test_surfaces_the_impossible_row():
    out = add_duration(coerce_types(RAW))
    assert (out.duration < 0).sum() == 1
    assert out.loc[out.duration < 0, "id"].iloc[0] == "A4"

test_coerce_counts_failures()
test_surfaces_the_impossible_row()
print("2 tests passed")                   # pytest finds and runs these itself

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
